# ThermoShift quickstart

Explore the bundled 50,003-decision dataset, select features, and compare logged
outcomes with the three available actions.

From the project root, install the walkthrough dependencies:

```bash
python -m pip install -e ".[hub,analysis,notebook]"
```

The cells below read `sample_dataset`. To explore another generated release, change
`DATA` in the first cell.

In [1]:
from contextlib import closing
from pathlib import Path

import pandas as pd
from datasets import load_dataset

from thermoshift import iter_pairs, validate, validated_manifest
from thermoshift.evaluation import evaluate_policy
from thermoshift.schema import feature_roles

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA = ROOT / "sample_dataset"
config, manifest = validated_manifest(DATA)
pd.DataFrame({"decisions": manifest["split_rows"]}).rename_axis("split")

,decisions
split,
test,4032
test_heatwave,1848
test_sensor,4139
train,35616
validation,4368


## Stream observed decisions

The `logged` configuration contains context, sampled actions, logging probabilities,
and factual outcomes. Column projection reads the selected fields; `take` limits
the records consumed.

In [2]:
records = load_dataset(
    str(DATA),
    name="logged",
    split="train",
    streaming=True,
    columns=[
        "row_id", "building_id", "step", "obs_temp_c", "action",
        "propensity", "y_next_temp_c", "y_energy_kwh",
    ],
)
pd.DataFrame(list(records.take(8)))

,row_id,building_id,step,obs_temp_c,action,propensity,y_next_temp_c,y_energy_kwh
0,0,0,0,25.413980,0,0.376731,25.615671,0.846533
1,1,0,1,25.556408,1,0.555121,24.184332,4.403686
2,2,0,2,24.351818,0,0.748566,24.301094,0.846533
3,3,0,3,24.550282,0,0.703491,24.385908,0.846533
4,4,0,4,24.505459,0,0.715530,24.466148,0.846533
5,5,0,5,NaN,0,0.708633,24.557121,0.846533
6,6,0,6,NaN,0,0.664248,24.726589,0.846533
7,7,0,7,NaN,0,0.598948,25.278074,0.846533


## Inspect release checks and split profiles

Full validation checks file integrity, row coverage, ordered trajectories, logging
probabilities, physical equations, sensor labels, and paired outcomes. This call
returns its findings without replacing the saved validation report.

In [3]:
report = validate(DATA, write_report=False)
assert report["status"] == "passed"
pd.DataFrame.from_dict(report["split_profiles"], orient="index")[
    ["rows", "null_sensor_rate", "outage_rate", "mean_outdoor_c", "mean_energy_kwh"]
].rename_axis("split")

,rows,null_sensor_rate,outage_rate,mean_outdoor_c,mean_energy_kwh
split,,,,,
test,4032,0.013145,0.002232,29.081080,2.252191
test_heatwave,1848,0.023810,0.012446,37.321291,3.296068
test_sensor,4139,0.349360,0.003624,28.696269,2.165191
train,35616,0.015836,0.004829,28.864007,2.081327
validation,4368,0.019231,0.001832,29.130229,1.956468


## Choose the task inputs

A decision policy uses the observed context. A transition model also receives the
action and predicts an outcome such as `y_next_temp_c`. The package provides the
corresponding feature lists.

In [4]:
roles = feature_roles()
pd.DataFrame([
    {"task": "decision policy", "features": ", ".join(roles["policy_features"])},
    {"task": "transition prediction", "features": ", ".join(roles["transition_features"])},
])

,task,features
0,decision policy,"hour, day_of_week, building_type, floor_area_m..."
1,transition prediction,"hour, day_of_week, building_type, floor_area_m..."


## Read matching potential outcomes

The oracle configuration stores each action's one-step outcome at the same state
and hourly disturbance. The factual target equals the branch selected by `action`.
The pair reader preserves row alignment and holds a dataset lease until it is
exhausted or closed.

In [5]:
with closing(iter_pairs(DATA, split="test", batch_size=8)) as batches:
    _, logged, oracle = next(batches)
    assert logged["row_id"].equals(oracle["row_id"])
    comparison = logged.select(["row_id", "action", "y_reward"]).to_pandas()
    potential = oracle.select([f"cf_reward_{action}" for action in range(3)]).to_pandas()
    comparison = pd.concat([comparison, potential], axis=1)
comparison

,row_id,action,y_reward,cf_reward_0,cf_reward_1,cf_reward_2
0,1344,1,-0.701304,-0.115247,-0.701304,-1.794699
1,1345,1,-0.646961,-0.047445,-0.646961,-1.305097
2,1346,0,-0.571037,-0.571037,-2.065110,-4.145368
3,1347,0,-0.477516,-0.477516,-1.884158,-3.876984
4,1348,0,-0.402246,-0.402246,-1.810452,-3.804842
5,1349,0,-0.284464,-0.284464,-1.491078,-3.283878
6,1350,0,-0.202916,-0.202916,-1.313965,-3.011199
7,1351,2,-2.345907,-0.112178,-0.941363,-2.345907


## Evaluate a fixed policy

The example policy selects a cooling level from the observed temperature gap.
IPS and SNIPS use the stored logging propensities; oracle evaluation directly
selects the same policy's potential outcome. These estimates describe one-step
reward on the logged state distribution. Uncertainty is clustered by building.

In [6]:
evaluation = evaluate_policy(DATA, split="test")
pd.DataFrame([
    {"estimate": name, **evaluation[name]}
    for name in ["ips", "snips", "oracle_policy_value", "logged_policy_value"]
]).set_index("estimate")

,value,se_building_cluster,ci95_normal_approx
estimate,,,
ips,-0.593095,0.067794,"[-0.7259719038379538, -0.4602187240983636]"
snips,-0.604877,0.067198,"[-0.7365855786411216, -0.4731674770928985]"
oracle_policy_value,-0.595999,0.064923,"[-0.7232475710300922, -0.4687496760810498]"
logged_policy_value,-0.724383,0.068578,"[-0.8587963112748584, -0.5899693635758334]"


## Generate and publish a release

Run the commands below from the project root to create a larger dataset:

```bash
python -m thermoshift init output/demo --rows 1000000 --seed 42
python -m thermoshift generate output/demo --workers 2
python -m thermoshift validate output/demo
```

See the [project README](../README.md) for loading and publishing,
[operations guide](../docs/generation.md) for workers and recovery, and
[model reference](../src/thermoshift/resources/DATASHEET.md) for the equations and
parameter distributions.